In [1]:
import torch
import numpy as np
from scipy.sparse import csr_matrix, load_npz
from torch.utils.data import DataLoader
from utils.Encoder_model import make_Encoder_model
from utils.loss import Combined_Loss
import torch
import numpy as np
from torch.utils.data import Dataset
from scipy.sparse import load_npz

from sklearn.metrics import ndcg_score

In [2]:
interactions = load_npz("data/data_train2.npz").tocsr()
user_embs_train = torch.load("data/user_embs_mu_train2.pt")
# user_embs_val = torch.load("data/user_embs_val_in.pt")
item_embs = torch.load("data/item_embs.pt")
item_bias = torch.load("data/item_bias.pt")

In [3]:
class UserItemDataset(torch.utils.data.Dataset):
    def __init__(self, user_embeddings: np.ndarray,
                       item_embeddings: np.ndarray,
                       interaction_matrix: csr_matrix,
                       sample_size: int = 500,
                       positive_sampling_bias: float = 0.9,
                       history_size: int=10):
        self.user_embeddings = user_embeddings
        self.item_embeddings = item_embeddings
        self.interactions = interaction_matrix.tocsr()
        self.sample_size = sample_size
        self.bias = positive_sampling_bias 
        self.num_items = item_embeddings.shape[0]
        self.num_users = user_embeddings.shape[0]
        self.history_size = history_size

    def __len__(self):
        return self.num_users

    def __getitem__(self, user_id):
        u_emb = self.user_embeddings[user_id]
        user_vector = self.interactions[user_id]

        pos_items = user_vector.indices
        pos_set = set(pos_items)
        
        if len(pos_items) >= self.history_size:
            history_items = np.random.choice(pos_items, size=self.history_size, replace=False)
        else:
            history_items = np.random.choice(pos_items, size=self.history_size, replace=True)

        excluded_items = set(history_items)

        probs = np.ones(self.num_items)
        probs[list(pos_set)] *= self.bias / (1 - self.bias)
        probs[list(excluded_items)] = 0 
        probs /= probs.sum()

        sampled_items = np.random.choice(
            np.arange(self.num_items),
            size=self.sample_size,
            replace=False,
            p=probs
        )

        all_items = np.concatenate([history_items, sampled_items])
        labels = np.array(
            [1.0] * self.history_size + 
            [1.0 if i in pos_set else 0.0 for i in sampled_items], 
            dtype=np.float32
        )

        item_embs = self.item_embeddings[all_items]  # shape: (H + K, D)
        user_embs = np.repeat(u_emb[None, :], self.history_size + self.sample_size, axis=0)  # (H + K, D)

        # Строим входные признаки без флага
        X = np.hstack([user_embs, item_embs])  # shape: (H + K, 2D)

        # Добавим флаг: история = 1, кандидаты = 0
        flags = np.concatenate([
            np.ones((self.history_size, 1), dtype=np.float32),
            np.zeros((self.sample_size, 1), dtype=np.float32)
        ], axis=0)  # shape: (H + K, 1)

        # Добавим флаг как новый столбец
        X = np.hstack([X, flags])  # shape: (H + K, 2D + 1)

        return torch.tensor(X, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32)



In [4]:
import torch
import numpy as np
from torch.utils.data import Dataset
from scipy.sparse import load_npz

class valDataset(Dataset):
    def __init__(self, user_emb, item_emb, item_bias, scores, valin_scores, history_size=10):
        self.users = torch.load(user_emb)
        self.items = torch.load(item_emb)
        self.bias = torch.load(item_bias)
        matrix = load_npz(scores)
        self.scores = matrix
        matrix = load_npz(valin_scores)
        self.valin_scores = self.get_nonzeros(matrix)
        self.history_size = history_size

    @staticmethod
    def get_nonzeros(data):
        return [data[i].indices.tolist() for i in range(data.shape[0])]

    def __len__(self):
        return self.scores.shape[0]

    def __getitem__(self, idx):
        user = self.users[idx]
        non_zeros = self.valin_scores[idx]

        if len(non_zeros) == 0:
            history_items = []
        elif len(non_zeros) >= self.history_size:
            history_items = np.random.choice(non_zeros, size=self.history_size, replace=False).tolist()
        else:
            history_items = np.random.choice(non_zeros, size=self.history_size, replace=True).tolist()

        exclude_items = set(history_items)
        i_scores = user @ self.items.T + self.bias
        i_scores[non_zeros] = -1e10
        i_scores[list(exclude_items)] = -1e10

        topk_scores, topk_items = torch.topk(i_scores, 100)
        item_embs = self.items[topk_items]

        if history_items:
            history_embs = self.items[history_items]
            history_scores = torch.ones(len(history_items), dtype=torch.float32)
            user_history = user.unsqueeze(0).repeat(len(history_items), 1)
            history_X = torch.cat([user_history, history_embs], dim=-1)
        else:
            history_X = torch.empty((0, self.users.shape[1] + self.items.shape[1]), dtype=torch.float32)
            history_scores = torch.empty((0,), dtype=torch.float32)

        user_topk = user.unsqueeze(0).repeat(100, 1)
        topk_X = torch.cat([user_topk, item_embs], dim=-1)
        topk_item_ids = topk_items.cpu().numpy()
        real_scores = torch.tensor(self.scores[idx, topk_item_ids].toarray(), dtype=torch.float32).squeeze()

        # Добавляем признак "история = 1"
        flag_history = torch.ones((history_X.size(0), 1), dtype=torch.float32)
        history_X = torch.cat([history_X, flag_history], dim=1)

        # Добавляем признак "topk = 0"
        flag_topk = torch.zeros((topk_X.size(0), 1), dtype=torch.float32)
        topk_X = torch.cat([topk_X, flag_topk], dim=1)

        X = torch.cat([history_X, topk_X], dim=0)
        labels = torch.cat([history_scores, real_scores], dim=0)
        topk_scores = torch.cat([torch.ones_like(history_scores), topk_scores], dim=0)

        return X, labels, topk_scores


In [5]:
dataset = UserItemDataset(
    user_embeddings=user_embs_train,
    item_embeddings=item_embs,
    interaction_matrix=interactions,
    sample_size=500,
    positive_sampling_bias=0.95,
    history_size=100
)

val_dataset = valDataset(
    user_emb="data/user_embs_mu_valid_in.pt",
    item_emb="data/item_embs.pt",
    item_bias="data/item_bias.pt",
    scores="data/data_valid_out.npz",
    valin_scores="data/data_valid_in.npz"
)


loader = DataLoader(dataset, batch_size=512, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
history_train = dataset.history_size
history_val = val_dataset.history_size

In [6]:
val_dataset[0]

(tensor([[-0.1457, -0.2198,  0.1147,  ..., -0.1774, -0.1904,  1.0000],
         [-0.1457, -0.2198,  0.1147,  ...,  0.2422, -0.0720,  1.0000],
         [-0.1457, -0.2198,  0.1147,  ...,  0.0602, -0.0545,  1.0000],
         ...,
         [-0.1457, -0.2198,  0.1147,  ...,  0.1814, -0.1139,  0.0000],
         [-0.1457, -0.2198,  0.1147,  ...,  0.0300, -0.2728,  0.0000],
         [-0.1457, -0.2198,  0.1147,  ...,  0.0759, -0.1146,  0.0000]]),
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 1., 0.,
         0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1.,
         0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0.]),
 tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.00

In [ ]:
model = make_Encoder_model(d_model=256, 
                           n_heads=2,
                           n_layers=2,
                           output_dim=2,
                           dropout_rate=0.1,
                           ffn_hidden=512,
                           input_dim=401
                           )

loss_fn = Combined_Loss(
    num_of_labels=2
)

device = "cuda"
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [8]:
import numpy as np
from sklearn.metrics import ndcg_score

for epoch in range(15):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        scores = model(X_batch)
        
        scores_cut = scores[:,history_train:]
        y_cut = y_batch[:,history_train:]
        loss = loss_fn(scores_cut, y_cut)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    ndcg_score_rerank = []
    ndcg_score_rank = []

    model.eval()
    for X_val, y_true, rank_scores in val_loader:
        X_val = X_val.to(device)
        y_true = y_true.to(device)

        scores_pred = model(X_val)
        scores_pred = torch.softmax(scores_pred, dim=-1)[:, :, 1]

        for user in range(X_val.size(0)):
            y_true_np = y_true[user][history_val:].cpu().numpy().reshape(1, -1)
            pred_np = scores_pred[user][history_val:].detach().cpu().numpy().reshape(1, -1)
            rank_np = rank_scores[user][history_val:].detach().cpu().numpy().reshape(1, -1)
            ndcg_score_rerank.append(ndcg_score(list(y_true_np), list(pred_np), k=100))
            ndcg_score_rank.append(ndcg_score(list(y_true_np), list(rank_np), k=100))

    print(f"Epoch {epoch+1}: train loss = {total_loss:.4f}")
    print(f"Epoch {epoch+1}: NDCG@100 rerank = {np.mean(ndcg_score_rerank):.3f}")
    print(f"Epoch {epoch+1}: NDCG@100 baseline = {np.mean(ndcg_score_rank):.3f}")


RuntimeError: mat1 and mat2 shapes cannot be multiplied (307200x401 and 400x256)